# Module 1: Building a Multi-Agent Customer Service System

## Overview

In this module, you'll build a production-ready multi-agent system for e-commerce customer service using the Strands Agent SDK with **MCP (Model Context Protocol)** tools. The system uses a **cost-optimized architecture** with:

- **Orchestrator Agent** (Claude Sonnet 4.5): Handles intent classification and complex reasoning
- **Order Agent** (Claude Haiku 4.5): Handles order inquiries via MCP tools
- **Product Agent** (Claude Haiku 4.5): Handles product search via MCP tools
- **Account Agent** (Claude Haiku 4.5): Handles account management (native tools)

### MCP Architecture
The Order and Product agents use **MCP (Model Context Protocol)** to connect to separate MCP servers that expose tools. This provides:
- **Decoupling**: Tools run as separate processes
- **Reusability**: MCP servers can be shared across applications
- **Standardization**: Follows the MCP protocol for tool discovery

### Learning Objectives
1. Build specialized agents with Strands SDK and MCP tools
2. Create an orchestrator for intelligent request routing
3. Understand cost optimization with mixed LLM strategy
4. Test the multi-agent system with various queries

### Time: ~60 minutes

## Step 1: Environment Setup

First, let's set up our environment and verify we have access to the required resources.

In [1]:
# Install dependencies if needed
!pip install -q strands-agents strands-agents-tools boto3 mcp


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


In [2]:
import boto3
import json
import os
import sys
from datetime import datetime

# Get AWS region
session = boto3.Session()
REGION = session.region_name or 'us-west-2'
print(f"AWS Region: {REGION}")

# Set environment variables for tools
os.environ['AWS_REGION'] = REGION

# Get table names from SSM (set by workshop infrastructure)
ssm = boto3.client('ssm', region_name=REGION)

try:
    os.environ['ORDERS_TABLE_NAME'] = ssm.get_parameter(Name='ecommerce-workshop-orders-table')['Parameter']['Value']
    os.environ['ACCOUNTS_TABLE_NAME'] = ssm.get_parameter(Name='ecommerce-workshop-accounts-table')['Parameter']['Value']
    os.environ['PRODUCTS_TABLE_NAME'] = ssm.get_parameter(Name='ecommerce-workshop-products-table')['Parameter']['Value']
    print(f"✓ Orders Table: {os.environ['ORDERS_TABLE_NAME']}")
    print(f"✓ Accounts Table: {os.environ['ACCOUNTS_TABLE_NAME']}")
    print(f"✓ Products Table: {os.environ['PRODUCTS_TABLE_NAME']}")
except Exception as e:
    print(f"Note: Could not retrieve SSM parameters ({e})")
    print("Using default table names - ensure infrastructure is deployed")
    os.environ['ORDERS_TABLE_NAME'] = 'ecommerce-workshop-orders'
    os.environ['ACCOUNTS_TABLE_NAME'] = 'ecommerce-workshop-accounts'
    os.environ['PRODUCTS_TABLE_NAME'] = 'ecommerce-workshop-products'

AWS Region: us-west-2
✓ Orders Table: ecommerce-workshop-orders
✓ Accounts Table: ecommerce-workshop-accounts
✓ Products Table: ecommerce-workshop-products


## Step 2: Understanding MCP Tools

The Order and Product agents use **MCP (Model Context Protocol)** to access tools. Instead of importing Python functions directly, the agents connect to MCP servers that expose tools.

### MCP Server Architecture

```
Agent (Haiku)  ──MCPClient──►  MCP Server  ──►  DynamoDB
                  (stdio)      (FastMCP)
```

### MCP Servers in this Module

| Server | File | Tools |
|--------|------|-------|
| Order Service | `mcp_servers/order_mcp_server.py` | get_order_status, track_shipment, process_return, modify_order, get_customer_orders |
| Product Service | `mcp_servers/product_mcp_server.py` | search_products, get_product_details, check_inventory, get_product_recommendations, compare_products, get_return_policy |

### How Agents Connect to MCP

```python
from strands.tools.mcp import MCPClient
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client

# Configure MCP server connection
server_params = StdioServerParameters(
    command="python",
    args=["mcp_servers/order_mcp_server.py"]
)

# Connect and get tools
mcp_client = MCPClient(lambda: stdio_client(server_params))
mcp_client.__enter__()
tools = mcp_client.list_tools_sync()

# Create agent with MCP tools
agent = Agent(model=model, tools=tools)
```

In [6]:
# Verify MCP servers exist
# Add agents to path
sys.path.insert(0, 'agents')
import os
from pathlib import Path

mcp_servers_dir = Path('mcp_servers')
order_server = mcp_servers_dir / 'order_mcp_server.py'
product_server = mcp_servers_dir / 'product_mcp_server.py'

print("MCP Servers:")
print(f"  Order Server: {order_server} - {'✓ exists' if order_server.exists() else '✗ missing'}")
print(f"  Product Server: {product_server} - {'✓ exists' if product_server.exists() else '✗ missing'}")

MCP Servers:
  Order Server: mcp_servers/order_mcp_server.py - ✓ exists
  Product Server: mcp_servers/product_mcp_server.py - ✓ exists


## Step 3: Build Specialized Agents

Now let's create our specialized agents. The Order and Product agents use **MCP tools** from separate MCP servers, while the Account agent uses native tools.

The MCP servers are located in `mcp_servers/` directory:
- `order_mcp_server.py` - Exposes order management tools
- `product_mcp_server.py` - Exposes product catalog tools

In [4]:
from strands import Agent
from strands.models import BedrockModel

# Model configuration using global cross-region inference profiles
HAIKU_MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
SONNET_MODEL_ID = "global.anthropic.claude-sonnet-4-5-20250929-v1:0"

# Pricing per 1M tokens (approximate, check AWS pricing page for latest)
# https://aws.amazon.com/bedrock/pricing/
HAIKU_INPUT_PRICE = 0.80    # $0.80 per 1M input tokens
HAIKU_OUTPUT_PRICE = 4.00   # $4.00 per 1M output tokens
SONNET_INPUT_PRICE = 3.00   # $3.00 per 1M input tokens  
SONNET_OUTPUT_PRICE = 15.00 # $15.00 per 1M output tokens

print(f"Haiku Model: {HAIKU_MODEL_ID}")
print(f"Sonnet Model: {SONNET_MODEL_ID}")
print("\nOrder and Product agents use MCP tools via MCPClient")

Haiku Model: global.anthropic.claude-haiku-4-5-20251001-v1:0
Sonnet Model: global.anthropic.claude-sonnet-4-5-20250929-v1:0

Order and Product agents use MCP tools via MCPClient


### 3.1 Create Order Agent

In [7]:
from order_agent import create_order_agent

order_agent = create_order_agent(region=REGION)
print(f"Order Agent created with model: {HAIKU_MODEL_ID}")
print("Tools loaded via MCP from order_mcp_server.py")

Order Agent created with model: global.anthropic.claude-haiku-4-5-20251001-v1:0
Tools loaded via MCP from order_mcp_server.py


In [8]:
# Test Order Agent directly
response = order_agent("What's the status of order ORD-2024-10002?")
print("Order Agent Response:")
print(response)


Tool #1: get_order_status
Great! Here's the status of your order:

**Order ID:** ORD-2024-10002  
**Status:** Shipped ✓

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Delivery Address:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is on its way! You can track your shipment using the UPS tracking number provided. Would you like me to get more detailed tracking information, or is there anything else I can help you with?Order Agent Response:
Great! Here's the status of your order:

**Order ID:** ORD-2024-10002  
**Status:** Shipped ✓

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Informa

### 3.2 Create Product Agent

In [9]:
from product_agent import create_product_agent

product_agent = create_product_agent(region=REGION)
print(f"Product Agent created with model: {HAIKU_MODEL_ID}")
print("Tools loaded via MCP from product_mcp_server.py")

Product Agent created with model: global.anthropic.claude-haiku-4-5-20251001-v1:0
Tools loaded via MCP from product_mcp_server.py


In [10]:
# Test Product Agent directly
response = product_agent("Do you have any wireless headphones under $100?")
print("Product Agent Response:")
print(response)

I'll search for wireless headphones under $100 for you.
Tool #1: search_products
Great! I found one wireless headphone option under $100:

**Wireless Bluetooth Headphones - $79.99** (PROD-001)
- **Type:** Over-ear headphones
- **Key Features:**
  - Active noise cancellation
  - 40-hour battery life
  - Comfortable memory foam ear cushions
  - Premium quality
- **Availability:** In stock ✓

This is a great value option at under $80! The 40-hour battery life is excellent for extended use.

I also found another option that's slightly above your budget:
- **Noise Canceling Earbuds - $149.99** (PROD-055) - True wireless earbuds with adaptive noise cancellation and 8-hour battery per charge

Would you like more details about the $79.99 headphones, or would you like me to check inventory status or help you compare them with other options?Product Agent Response:
Great! I found one wireless headphone option under $100:

**Wireless Bluetooth Headphones - $79.99** (PROD-001)
- **Type:** Over-ear 

### 3.3 Create Account Agent

In [11]:
from account_agent import create_account_agent

account_agent = create_account_agent(region=REGION)
print(f"Account Agent created with model: {HAIKU_MODEL_ID}")

Account Agent created with model: global.anthropic.claude-haiku-4-5-20251001-v1:0


In [12]:
# Test Account Agent directly
response = account_agent("What are the benefits of Gold membership?")
print("Account Agent Response:")
print(response)


Tool #1: get_membership_benefits
Great question! Here are the benefits of **Gold membership**:

✨ **Gold Membership Benefits:**
- **Free Shipping** on orders over $25 (compared to $50 for Standard)
- **45-Day Return Window** for returns and exchanges
- **Early Access to Sales** - Get first dibs on exclusive deals
- **Earn 1.5x Points** - Earn 1.5 reward points per $1 spent (vs. standard rate)
- **Birthday Discount** - Receive 15% off during your birthday month

**Why Gold is a great choice:**
Gold membership is perfect if you shop regularly and want to save on shipping costs. The lower $25 free shipping threshold means you'll qualify for free shipping more often, and the early sale access helps you get the best deals before they sell out.

If you're interested in upgrading to Gold or have any other questions about your membership, I'm happy to help! Would you also like to know about our **Platinum tier** for even more premium benefits?Account Agent Response:
Great question! Here are t

## Step 4: Create the Orchestrator

The Orchestrator uses Claude Sonnet 4.5 (larger model) for complex reasoning and routes requests to specialized agents.

In [13]:
from orchestrator import MultiAgentCustomerService

# Create the multi-agent system
customer_service = MultiAgentCustomerService(region=REGION)
print("Multi-Agent Customer Service System initialized!")
print(f"- Orchestrator: {SONNET_MODEL_ID}")
print(f"- Order Agent: {HAIKU_MODEL_ID} (MCP tools)")
print(f"- Product Agent: {HAIKU_MODEL_ID} (MCP tools)")
print(f"- Account Agent: {HAIKU_MODEL_ID} (native tools)")

Multi-Agent Customer Service System initialized!
- Orchestrator: global.anthropic.claude-sonnet-4-5-20250929-v1:0
- Order Agent: global.anthropic.claude-haiku-4-5-20251001-v1:0 (MCP tools)
- Product Agent: global.anthropic.claude-haiku-4-5-20251001-v1:0 (MCP tools)
- Account Agent: global.anthropic.claude-haiku-4-5-20251001-v1:0 (native tools)


## Step 5: Test the Multi-Agent System

Let's test various customer scenarios to see how the orchestrator routes requests.

In [14]:
# Test Case 1: Order inquiry (should route to Order Agent)
print("="*60)
print("Test Case 1: Order Inquiry")
print("="*60)

response = customer_service.chat("Where is my order ORD-2024-10002?")
print(f"Customer: Where is my order ORD-2024-10002?")
print(f"\nAgent: {response}")

Test Case 1: Order Inquiry

Tool #1: delegate_to_order_agent
I'll check the status and tracking information for your order.
Tool #1: get_order_status

Tool #2: track_shipment
Great news! Your order **ORD-2024-10002** is on its way! 📦

**Order Status:** Shipped

**Shipping Details:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Estimated Delivery:** January 8, 2025
- **Shipping To:** Los Angeles, CA 90001

**Items in Your Order:**
- Smart Watch Pro (1x) - $299.99
- Watch Band - Leather (2x) - $29.99 each

**Order Total:** $359.97

You can track your package in real-time using your UPS tracking number here: https://www.ups.com/track?tracknum=1Z999AA10123456784

Is there anything else you'd like to know about your order?Great news! Your order **ORD-2024-10002** is on its way! 📦

**Order Status:** Shipped

**Shipping Details:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Estimated Delivery:** January 8, 2025
- **Shipping To:** Los Angeles, CA 90001


In [15]:
# Test Case 2: Product search (should route to Product Agent)
print("="*60)
print("Test Case 2: Product Search")
print("="*60)

response = customer_service.chat("I'm looking for a good gaming keyboard with RGB lighting")
print(f"Customer: I'm looking for a good gaming keyboard with RGB lighting")
print(f"\nAgent: {response}")

Test Case 2: Product Search

Tool #2: delegate_to_product_agent
I'll search for gaming keyboards with RGB lighting for you.
Tool #1: search_products
Great! I found a gaming keyboard with RGB lighting. Let me get more detailed information about it.
Tool #2: get_product_details

Tool #3: check_inventory
Perfect! Here's what I found for you:

## **Gaming Mechanical Keyboard RGB** (PROD-077)
**Price:** $159.99

### Key Features:
- **Lighting:** Per-key RGB with 16.8M colors - fully customizable
- **Switches:** Hot-swappable Cherry MX compatible switches (easy to replace)
- **Layout:** Full-size mechanical keyboard
- **Connection:** USB-C detachable cable for portability
- **Performance:** Full N-key rollover (anti-ghosting) for responsive gaming
- **Comfort:** Detachable wrist rest included
- **Controls:** Dedicated media keys + volume wheel

### Additional Details:
- **Rating:** 4.6 out of 5 stars
- **Availability:** ✅ In Stock (60 units available)
- **Warranty:** 2 years
- **Return Polic

In [16]:
# Test Case 3: Account management (should route to Account Agent)
print("="*60)
print("Test Case 3: Account Management")
print("="*60)

response = customer_service.chat("I need to reset my password for john.smith@email.com")
print(f"Customer: I need to reset my password for john.smith@email.com")
print(f"\nAgent: {response}")

Test Case 3: Account Management

Tool #3: delegate_to_account_agent
I'll initiate a password reset for your account right away.
Tool #1: initiate_password_reset
Perfect! I've initiated the password reset process for john.smith@email.com. Here's what you need to know:

**Password Reset Details:**
- ✓ A password reset link has been sent to john.smith@email.com
- ⏱️ The link will expire in **1 hour**, so please act quickly
- 📧 Check your email inbox (and spam/junk folder just in case)
- 🔒 For security reasons, we don't confirm whether an account exists with this email

**Next Steps:**
1. Check your email for the reset link
2. Click the link to create a new password
3. Make sure your new password is strong and unique
4. You'll be able to log in with your new password immediately

If you don't receive the email within a few minutes, please check your spam folder. If you still don't see it, let me know and I can help troubleshoot further!Perfect! I've initiated the password reset process for

In [17]:
# Test Case 4: Complex multi-domain query (should use multiple agents)
print("="*60)
print("Test Case 4: Complex Multi-Domain Query")
print("="*60)

response = customer_service.chat(
    "I want to return order ORD-2024-10001 because the headphones don't fit well. "
    "Can you recommend a different pair that might be more comfortable?"
)
print(f"Customer: I want to return order ORD-2024-10001 because the headphones don't fit well. Can you recommend a different pair that might be more comfortable?")
print(f"\nAgent: {response}")

Test Case 4: Complex Multi-Domain Query

Tool #4: delegate_to_order_agent
I'll process a return request for your order ORD-2024-10001.
Tool #3: process_return
I'm sorry, but I'm unable to process a return for order **ORD-2024-10001** at this time. 

**Reason:** The return window has expired. Our standard return policy allows returns within 30 days of purchase, and your order was placed 401 days ago.

**What you can do:**
Since this is outside our standard return window, I'd recommend contacting our customer service team directly. They may be able to help you with options or accommodations depending on the circumstances. A human agent might be able to review your situation and explore alternative solutions.

Is there anything else I can help you with?
Tool #5: delegate_to_product_agent
I'll get some personalized recommendations for comfortable headphones based on your previous fit issues.
Tool #4: get_product_recommendations
Great! I found some comfortable headphone options. Let me get 

In [18]:
# Test Case 5: Inventory check
print("="*60)
print("Test Case 5: Inventory Check")
print("="*60)

response = customer_service.chat("Is the 4K webcam PROD-088 in stock?")
print(f"Customer: Is the 4K webcam PROD-088 in stock?")
print(f"\nAgent: {response}")

Test Case 5: Inventory Check

Tool #6: delegate_to_product_agent

Tool #7: check_inventory
Unfortunately, the 4K webcam (PROD-088) is **currently out of stock**. 

**Restock Information:**
- **Expected Restock Date:** January 15, 2025
- **Current Stock:** 0 units available

Would you like me to:
1. **Get details about PROD-088** so you know exactly what to expect when it's back in stock?
2. **Search for alternative 4K webcams** that are currently available?
3. **Help you find other camera options** in our catalog?

Let me know how I can help!Unfortunately, the 4K webcam (PROD-088) is **currently out of stock**. 

**Restock Information:**
- **Expected Restock Date:** January 15, 2025
- **Current Stock:** 0 units available

Would you like me to:
1. **Get details about PROD-088** so you know exactly what to expect when it's back in stock?
2. **Search for alternative 4K webcams** that are currently available?
3. **Help you find other camera options** in our catalog?

Let me know how I can 

## Step 6: Cost Analysis

Let's analyze the cost difference between using a single large model vs. our mixed architecture.

In [19]:
# Simulated token usage for a typical customer service session
# Based on average query complexity

def calculate_cost_comparison():
    """Compare cost between all-Sonnet vs mixed architecture"""
    
    # Average tokens per interaction (approximate)
    orchestrator_input = 800   # System prompt + query + tool descriptions
    orchestrator_output = 200  # Routing decision + brief response
    sub_agent_input = 600      # System prompt + delegated query + tools
    sub_agent_output = 400     # Detailed response
    
    # Scenario: 1000 customer interactions
    num_interactions = 1000
    
    # All-Sonnet Architecture (single large model)
    all_sonnet_input = (orchestrator_input + sub_agent_input) * num_interactions
    all_sonnet_output = (orchestrator_output + sub_agent_output) * num_interactions
    all_sonnet_cost = (
        (all_sonnet_input / 1_000_000) * SONNET_INPUT_PRICE +
        (all_sonnet_output / 1_000_000) * SONNET_OUTPUT_PRICE
    )
    
    # Mixed Architecture (Sonnet orchestrator + Haiku sub-agents)
    mixed_orchestrator_input = orchestrator_input * num_interactions
    mixed_orchestrator_output = orchestrator_output * num_interactions
    mixed_subagent_input = sub_agent_input * num_interactions
    mixed_subagent_output = sub_agent_output * num_interactions
    
    mixed_cost = (
        # Orchestrator (Sonnet 4.5)
        (mixed_orchestrator_input / 1_000_000) * SONNET_INPUT_PRICE +
        (mixed_orchestrator_output / 1_000_000) * SONNET_OUTPUT_PRICE +
        # Sub-agents (Haiku 4.5)
        (mixed_subagent_input / 1_000_000) * HAIKU_INPUT_PRICE +
        (mixed_subagent_output / 1_000_000) * HAIKU_OUTPUT_PRICE
    )
    
    savings = all_sonnet_cost - mixed_cost
    savings_percent = (savings / all_sonnet_cost) * 100
    
    return {
        'num_interactions': num_interactions,
        'all_sonnet_cost': round(all_sonnet_cost, 2),
        'mixed_cost': round(mixed_cost, 2),
        'savings': round(savings, 2),
        'savings_percent': round(savings_percent, 1)
    }

cost_analysis = calculate_cost_comparison()

print("Cost Comparison Analysis")
print("="*50)
print(f"Scenario: {cost_analysis['num_interactions']:,} customer interactions")
print()
print(f"All-Sonnet Architecture:   ${cost_analysis['all_sonnet_cost']:.2f}")
print(f"Mixed Architecture:        ${cost_analysis['mixed_cost']:.2f}")
print()
print(f"Savings:                   ${cost_analysis['savings']:.2f} ({cost_analysis['savings_percent']}%)")
print()
print("Note: Mixed architecture uses Sonnet 4.5 only for orchestration,")
print("      while Haiku 4.5 handles specialized tasks.")

Cost Comparison Analysis
Scenario: 1,000 customer interactions

All-Sonnet Architecture:   $13.20
Mixed Architecture:        $7.48

Savings:                   $5.72 (43.3%)

Note: Mixed architecture uses Sonnet 4.5 only for orchestration,
      while Haiku 4.5 handles specialized tasks.


## Summary

In this module, you built a multi-agent customer service system using **MCP tools** that:

1. **Uses MCP tools** - Order and Product agents connect to MCP servers for tool access
2. **Optimizes costs** by using smaller models for specialized tasks
3. **Routes intelligently** using a larger model for orchestration only
4. **Handles complex queries** that span multiple domains

### MCP Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    Orchestrator (Sonnet)                     │
│                   - Intent classification                    │
│                   - Request routing                          │
└─────────────────────┬───────────────┬───────────────────────┘
                      │               │
        ┌─────────────▼───┐   ┌───────▼─────────────┐
        │  Order Agent    │   │   Product Agent     │
        │   (Haiku)       │   │     (Haiku)         │
        │   MCP Tools     │   │     MCP Tools       │
        └────────┬────────┘   └─────────┬───────────┘
                 │                      │
        ┌────────▼────────┐   ┌─────────▼───────────┐
        │ order_mcp_server│   │ product_mcp_server  │
        │    (FastMCP)    │   │     (FastMCP)       │
        └────────┬────────┘   └─────────┬───────────┘
                 │                      │
        ┌────────▼────────┐   ┌─────────▼───────────┐
        │    DynamoDB     │   │     DynamoDB        │
        │  Orders Table   │   │   Products Table    │
        └─────────────────┘   └─────────────────────┘
```

### Key Files

| File | Description |
|------|-------------|
| `mcp_servers/order_mcp_server.py` | MCP server for order tools |
| `mcp_servers/product_mcp_server.py` | MCP server for product tools |
| `agents/order_agent.py` | Order Agent with MCP client |
| `agents/product_agent.py` | Product Agent with MCP client |
| `agents/orchestrator.py` | Multi-agent orchestrator |

### Key Takeaways

- **Cost Optimization**: Using Haiku 4.5 for sub-agents instead of Sonnet 4.5 everywhere can save ~50% on LLM costs
- **MCP Architecture**: Tools are exposed via MCP servers, enabling reusability and decoupling
- **Orchestration**: The orchestrator handles intent classification and complex reasoning
- **Scalability**: This architecture scales well as you add more specialized agents

### Next Steps

In **Module 2**, we'll evaluate this system with a comprehensive test dataset and establish baseline metrics for production monitoring.

In [ ]:
# Clean up MCP connections before ending
order_agent.cleanup()
product_agent.cleanup()
customer_service.cleanup()
print("MCP connections cleaned up")

# Save region for use in next module
%store REGION
print("Session data saved for Module 2!")